# Phase 3 — New Dataset Ingestion

Ingest all datasets listed in `data/README.md § Phase 3 datasets` into the
`laminlabs/pertdata` LaminDB instance as canonical triplets
(`obs.parquet / X.h5ad / var.parquet`).

All ingestion functions live in:
- `tools/ingest_phase3_scrna.py` — scRNA-seq datasets (PRISM, T-cell GWPS, VIPerturbSeq, PRoPER-seq source gate, Sanger dual-guide, Arc VCC)
- `tools/ingest_phase3_bulk.py` — bulk / sensitivity datasets (Broad PRISM, Sanger GDSC, Sanger SCORE, DepMap CCLE)
- `tools/ingest_xatlas_orion.py` — XAtlas / Orion (already implemented)

Uncomment and run each section's cell when ready.

## Sections
1. XAtlas / Orion — `tools/ingest_xatlas_orion.py` (already written)
2. PRISM Perturb-seq Collection (~36 h5ads from Google Drive)
3. T-cell Genome-Wide Perturb-seq (GSE314342, AWS S3)
4. VIPerturbSeq (Zenodo 18460279)
5. PRoPER-seq / ProPer-seq 2026 (source TBD)
6. Sanger Dual-guide KO in CRC (Figshare 25533091)
7. Arc VCC Perturbations
8. Broad PRISM Repurposing (bulk sensitivity)
9. Sanger GDSC (bulk sensitivity)
10. Sanger SCORE CRISPR KO (bulk gene effect)
11. DepMap CCLE (bulk RNA expression)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "..")

import lamindb as ln

from tools.ingest_xatlas_orion import run_xatlas_orion_pipeline
from tools.ingest_phase3_scrna import (
    download_prism_collection,
    ingest_prism_collection,
    download_tcell_gwps,
    ingest_tcell_gwps,
    download_viperturb,
    ingest_viperturb,
    check_properseq_in_lamin,
    ingest_properseq,  # raises until the real 2026 source is found
    ingest_sanger_dualguide,
    ingest_arc_vcc,
)
from tools.ingest_phase3_bulk import (
    ingest_broad_prism,
    ingest_gdsc,
    ingest_sanger_score,
    ingest_depmap_ccle,
)

ln.connect("laminlabs/pertdata")
ln.track()

---
## 1. XAtlas / Orion (Figshare 29190726)

Two genome-wide CRISPRi Perturb-seq datasets in HCT116 (~350 GB) and HEK293T (~150 GB).
Pipeline already implemented in `tools/ingest_xatlas_orion.py`.

Download → backed-mode extract obs/var → save triplet chunks to Lamin.

In [ ]:
# run_xatlas_orion_pipeline(
#     working_dir=Path("data/main/xatlas_orion"),
#     verify_md5=True,
#     instance="laminlabs/pertdata",
#     dataset_prefix="xatlas/orion",
#     chunk_size=600_000,
#     overwrite=False,
# )

---
## 2. PRISM Perturb-seq Collection (~36 datasets)

A benchmark collection of ~36 public Perturb-seq h5ads assembled in:
> Genome-scale perturb-seq in primary human CD4+ T cells … (bioRxiv 2025.12.23.696273)

All h5ads are hosted in a public Google Drive folder:
`https://drive.google.com/drive/folders/1Y0Z19JhiTmTch65kvBNNMdVtosH6QHfi`

Each file uses a harmonised obs schema:
`perturbation`, `perturbation_type`, `cell_line`, `organism`, `n_counts`, `n_genes`,
`is_control`.

**Requirements:** `pip install gdown`

In [ ]:
# prism_files = download_prism_collection(output_dir=Path("data/main/prism_collection"))
# ingest_prism_collection(prism_files)

---
## 3. T-cell Genome-Wide Perturb-seq (GSE314342)

22 million primary human CD4+ T cells (4 donors × 3 stimulation conditions).
CRISPRi targeting all protein-coding genes.

**Access:**
```bash
aws s3 ls --no-sign-request s3://genome-scale-tcell-perturb-seq/marson2025_data/
aws s3 sync --no-sign-request s3://genome-scale-tcell-perturb-seq/marson2025_data/ data/main/tcell_gwps/
```

The processed data contains cell-level count matrices (.h5ad), pseudobulk matrices,
and DE estimates. We only need the cell-level h5ads for our triplet format.

**Requirements:** AWS CLI installed (`brew install awscli`)

In [ ]:
# tcell_files = download_tcell_gwps(output_dir=Path("data/main/tcell_gwps"))
# ingest_tcell_gwps(tcell_files)

---
## 4. VIPerturbSeq (Zenodo 18460279)

Genome-wide CRISPRi screens using a novel split-probe strategy (GuEST-List library).
Two workflows: unbiased genome-wide screen + phenotypically enriched (VIP).

**Paper:** Bradu et al. 2026, `https://www.biorxiv.org/content/10.64898/2026.02.12.705613v1.full`  
**Data:** `https://zenodo.org/records/18460279`

In [ ]:
# vip_files = download_viperturb(output_dir=Path("data/main/viperturb"))
# ingest_viperturb(vip_files)

---
## 5. PRoPER-seq / ProPer-seq 2026 (source TBD)

Wanted target: the 2026 probe-based Perturb-seq scRNA expression matrix.

The legacy GSE150818 chimeric-read-pair supplementary dataset is excluded from pert-gym and was archived from the active `jkobject` branch if present. Do not use it as a substitute.


In [ ]:
# check_properseq_in_lamin()  # confirms no sourced 2026 PRoPER-seq matrix yet
# ingest_properseq()  # intentionally raises until source is identified

---
## 6. Sanger Dual-guide KO in CRC (Figshare 25533091)

Next-generation dual-guide CRISPR system applied to genetic interaction screening
in colorectal cancer (CRC) cell lines. Published as a tRNA spacer dual-guide library.

**Paper:** Burgold et al., Nature Communications 2025, `10.1038/s41467-025-67256-9`  
**Data:** `https://figshare.com/articles/dataset/MAPPING_zip/25533091/1?file=45433417`

The `MAPPING.zip` archive contains genetic interaction score matrices
(pairwise guide-pair scores per cell line) and likely also per-cell count data.

In [ ]:
# ingest_sanger_dualguide()

---
## 7. Arc VCC Perturbations

Dataset released for the Arc Virtual Cell Challenge.  
**Portal:** `https://virtualcellchallenge.org/datasets`

> **TODO:** visit the portal to confirm download method and file format before
> implementing the ingestion function.

In [ ]:
# ingest_arc_vcc()  # raises NotImplementedError until portal is checked

---
## 8. Broad PRISM Repurposing (bulk drug sensitivity)

Multiplexed cell-line viability for ~4 686 small-molecule compounds × ~578 cancer cell lines,
measured as log fold change vs DMSO.

**Source:** `https://depmap.org/repurposing/`  
**DepMap portal:** download `Repurposing_Public_23Q2_Extended_Primary_Data_Matrix.csv`
(or the latest release)

Hybrid AnnData: control rows carry CCLE expression in X; drug-treated rows carry LFC in obs,
X is sparse zeros.  See `tools/ingest_phase3_bulk.py::broad_prism_to_anndata` for details.

In [ ]:
# Optionally pass the CCLE expression CSV to fill control-row X with gene expression:
# ingest_broad_prism(ccle_expr_file=Path("data/main/depmap_ccle/OmicsExpressionProteinCodingGenesTPMLogp1.csv"))
# ingest_broad_prism()  # without CCLE, control rows will have empty X

---
## 9. Sanger GDSC (GDSC1 + GDSC2)

Fitted dose-response IC50 values for hundreds of cancer drugs × ~1 000 cancer cell lines.

**Source:** `https://www.cancerrxgene.org/downloads/bulk_download`  
Two studies: **GDSC1** (Sanger + MGH) and **GDSC2** (improved Sanger protocol).

Hybrid AnnData: control rows carry CMP expression in X; drug-treated rows carry IC50/AUC in obs,
X is sparse zeros.  See `tools/ingest_phase3_bulk.py::gdsc_to_anndata` for details.

In [ ]:
# Optionally pass the CMP expression file to fill control-row X with gene expression:
# ingest_gdsc(expr_file=Path("data/main/sanger_score/rnaseq_all_20220624.csv.gz"))
# ingest_gdsc()  # without CMP expression, control rows will have empty X

---
## 10. Sanger SCORE CRISPR KO (Cell Model Passports)

Genome-wide CRISPR-Cas9 dropout screens in 323+ cancer cell lines.
Gene effect values are corrected for copy-number bias with CRISPRcleanR.

**Source:** `https://cellmodelpassports.sanger.ac.uk/downloads`  
**Documentation:** `https://depmap.sanger.ac.uk/documentation/datasets/wg-crispr-knockout/`

Stored as AnnData: `obs` = cell lines, `var` = genes, `X` = gene effect (Bayes Factor).

In [ ]:
# ingest_sanger_score()

---
## 11. DepMap CCLE (bulk RNA expression + proteomics)

Cancer Cell Line Encyclopedia bulk omics: RNA-seq (log₂ TPM+1), proteomics (MS),
mutations (MAF), and copy number.

**Source:** `https://depmap.org/portal/download/` (DepMap Public 25Q2 or latest)  
**Files needed:**
- `OmicsExpressionProteinCodingGenesTPMLogp1.csv` — RNA expression matrix
- `OmicsProteinExpressionLog2.csv` — proteomics matrix (optional)

Stored as AnnData: `obs` = cell lines, `var` = genes, `X` = log₂(TPM+1).

> **Note:** Set `DEPMAP_FIGSHARE_ARTICLE` in `tools/ingest_phase3_bulk.py` to the
> correct Figshare article ID for the current release before calling `ingest_depmap_ccle()`.

In [ ]:
# ingest_depmap_ccle()

---
## Summary

| # | Dataset | Type | Lamin prefix | Status |
|---|---------|------|-------------|--------|
| 1 | XAtlas/Orion (HCT116 + HEK293T) | scRNA CRISPRi | `xatlas/orion` | ⏳ run pipeline |
| 2 | PRISM collection (~36 studies) | scRNA CRISPRi/KO | `prism_collection/*` | ⏳ download from GDrive |
| 3 | T-cell GWPS (GSE314342) | scRNA CRISPRi | `tcell_gwps/*` | ⏳ sync from S3 |
| 4 | VIPerturbSeq (Zenodo 18460279) | scRNA CRISPRi | `viperturb/*` | ⏳ download from Zenodo |
| 5 | PRoPER-seq / ProPer-seq 2026 | scRNA probe-based Perturb-seq | `properseq` | 🚫 source TBD; legacy GSE150818 excluded |
| 6 | Sanger dual-guide CRC (Figshare 25533091) | scRNA dual-KO | `sanger_dual_guide_crc/*` | ⏳ download MAPPING.zip |
| 7 | Arc VCC perturbations | scRNA/other | `arc_vcc` | ❌ check portal |
| 8 | Broad PRISM repurposing | sensitivity | `broad_prism_repurposing` | ⏳ download CSV |
| 9 | Sanger GDSC (GDSC1 + GDSC2) | sensitivity | `sanger_gdsc/{gdsc1,gdsc2}` | ⏳ download Excel |
| 10 | Sanger SCORE CRISPR KO | gene effect | `sanger_score_crispr` | ⏳ check CMP page |
| 11 | DepMap CCLE | bulk RNA | `depmap_ccle/25q2` | ⏳ set Figshare article ID |